In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import pickle
import optuna
import warnings
import json

warnings.filterwarnings('ignore')

In [3]:
# ================== 0. 数据加载 ==================
def load_data_from_excel(lensdata_path: str, surfdata_path: str):
    """从 Excel 文件加载光学设计数据，返回 patent_list。"""
    lens_df = pd.read_excel(lensdata_path)
    surf_df = pd.read_excel(surfdata_path)

    grouped_surf = surf_df.groupby(['PATNUM', 'EXNUM', 'ZNUM'])

    patent_list = []
    for idx, lens_row in lens_df.iterrows():
        pat = int(lens_row['PATNUM'])
        ex = int(lens_row['EXNUM'])
        zn = int(lens_row['ZNUM'])

        efl = float(lens_row['EFL'])
        epd = float(lens_row['EPD'])
        max_fang = float(lens_row['MAXFANG'])
        max_img_height = float(lens_row['MAXRIMGHT'])
        bfl = float(lens_row['BFL'])
        num_elements = int(lens_row['NGLASELEM'])
        primary_wl = float(lens_row['WL1'])

        f_number = efl / epd if epd != 0 else 0.0

        features = {
            'EFL': efl,
            'F/#': f_number,
            'Max Field Angle': max_fang,
            'Image Height': max_img_height,
            'BFL': bfl,
            'Number of Elements': num_elements,
            'Primary Wavelength': primary_wl
        }

        try:
            surf_group = grouped_surf.get_group((pat, ex, zn))
        except KeyError:
            continue

        surfaces = []
        surf_sorted = surf_group.sort_values('SNUM')
        max_snum = surf_sorted['SNUM'].max()
        for _, srow in surf_sorted.iterrows():
            atype = str(srow['ATYPE']).strip().upper()
            curvature = float(srow['CUY'])
            thickness = float(srow['THI'])
            nd = float(srow['IND1']) if not pd.isna(srow['IND1']) else 0.0
            vd = float(srow['VNUM']) if not pd.isna(srow['VNUM']) else 0.0

            if atype == 'REFR':
                stype = 'glass'
            elif atype == 'AIR':
                if srow['SNUM'] == max_snum and abs(thickness) < 1e-6:
                    stype = 'image'
                else:
                    stype = 'air'
            else:
                stype = 'air'

            surfaces.append({
                'curvature': curvature,
                'thickness': thickness,
                'nd': nd,
                'vd': vd,
                'type': stype
            })
        patent_list.append({**features, 'surfaces': surfaces})
    return patent_list

In [4]:
# ================== 1. 数据集 ==================
type_to_idx = {'air': 0, 'glass': 1, 'image': 2, 'start': 3}
idx_to_type = {v: k for k, v in type_to_idx.items()}
num_gen_types = 3

class LensDataset(Dataset):
    def __init__(self, patent_list, max_len=35):
        self.patent_list = patent_list
        self.max_len = max_len
        self._prepare_statistics()

    def _prepare_statistics(self):
        input_keys = ['EFL', 'F/#', 'Max Field Angle', 'Image Height', 'BFL',
                      'Number of Elements', 'Primary Wavelength']
        inputs = []
        all_C, all_t, all_nd, all_vd = [], [], [], []

        for patent in self.patent_list:
            inp = [patent[k] for k in input_keys]
            inputs.append(inp)
            for surf in patent['surfaces']:
                all_C.append(surf['curvature'])
                all_t.append(surf['thickness'])
                all_nd.append(surf['nd'])
                all_vd.append(surf['vd'])

        inputs = np.array(inputs, dtype=np.float32)
        self.input_scaler = StandardScaler()
        self.input_scaler.fit(inputs)

        self.C_scaler = StandardScaler()
        self.t_scaler = StandardScaler()
        self.nd_scaler = StandardScaler()
        self.vd_scaler = StandardScaler()

        self.C_scaler.fit(np.array(all_C).reshape(-1, 1))
        self.t_scaler.fit(np.array(all_t).reshape(-1, 1))
        self.nd_scaler.fit(np.array(all_nd).reshape(-1, 1))
        self.vd_scaler.fit(np.array(all_vd).reshape(-1, 1))

    def __len__(self):
        return len(self.patent_list)

    def __getitem__(self, idx):
        patent = self.patent_list[idx]
        input_keys = ['EFL', 'F/#', 'Max Field Angle', 'Image Height', 'BFL',
                      'Number of Elements', 'Primary Wavelength']
        raw_input = np.array([patent[k] for k in input_keys], dtype=np.float32)
        inp_scaled = self.input_scaler.transform(raw_input.reshape(1, -1)).flatten()
        inp_tensor = torch.FloatTensor(inp_scaled)

        num_elements = int(patent['Number of Elements'])

        surfaces = patent['surfaces']
        seq_len = len(surfaces)
        seq = np.zeros((seq_len, 5), dtype=np.float32)
        for i, surf in enumerate(surfaces):
            type_idx_val = type_to_idx[surf['type']]
            c = self.C_scaler.transform([[surf['curvature']]])[0, 0]
            t_val = self.t_scaler.transform([[surf['thickness']]])[0, 0]
            nd = self.nd_scaler.transform([[surf['nd']]])[0, 0]
            vd = self.vd_scaler.transform([[surf['vd']]])[0, 0]
            seq[i] = [type_idx_val, c, t_val, nd, vd]

        padded_seq = np.zeros((self.max_len, 5), dtype=np.float32)
        padded_seq[:seq_len] = seq
        mask = np.zeros(self.max_len, dtype=bool)
        mask[:seq_len] = True

        padded_seq_tensor = torch.FloatTensor(padded_seq)
        mask_tensor = torch.BoolTensor(mask)
        length = seq_len

        return inp_tensor, padded_seq_tensor, mask_tensor, length, num_elements


In [5]:
# ================== 2. 模型定义 ==================
class ConditionEncoder(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=256, cond_dim=128, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, cond_dim)
        )
        self.h0 = nn.Linear(cond_dim, num_layers * hidden_dim)
        self.c0 = nn.Linear(cond_dim, num_layers * hidden_dim)

    def forward(self, x):
        cond = self.net(x)
        h0 = self.h0(cond)
        c0 = self.c0(cond)
        h0 = h0.view(self.num_layers, -1, h0.size(-1) // self.num_layers)
        c0 = c0.view(self.num_layers, -1, c0.size(-1) // self.num_layers)
        return cond, (h0, c0)

class SurfaceDecoder(nn.Module):
    def __init__(self, cond_dim=128, hidden_dim=256, num_layers=2, dropout=0.1, max_elements=15):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.max_elements = max_elements
        self.type_embed = nn.Embedding(4, 16)
        self.remain_embed = nn.Linear(1, 4)
        input_dim = 16 + 4 + cond_dim + 4
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.type_head = nn.Linear(hidden_dim, 3)
        self.reg_head = nn.Linear(hidden_dim, 4)

    def forward_step(self, prev_surf, prev_hidden, cond, remaining):
        B = prev_surf.size(0)
        type_idx = prev_surf[:, 0].long()
        type_emb = self.type_embed(type_idx)
        reg_val = prev_surf[:, 1:]

        rem_norm = remaining.float() / self.max_elements
        rem_norm = rem_norm.unsqueeze(1)
        rem_feat = self.remain_embed(rem_norm)

        if cond.dim() == 3:
            cond = cond.squeeze(1)

        lstm_input = torch.cat([type_emb, reg_val, cond, rem_feat], dim=-1)
        lstm_input = lstm_input.unsqueeze(1)
        output, next_hidden = self.lstm(lstm_input, prev_hidden)
        type_logits = self.type_head(output.squeeze(1))
        reg_pred = self.reg_head(output.squeeze(1))
        return type_logits, reg_pred, next_hidden

    def forward(self, init_hidden, cond, targets=None, teacher_forcing_ratio=0.5, max_len=30, num_elements=None):
        B = cond.size(0)
        device = cond.device
        start_surf = torch.zeros(B, 5, device=device)
        start_surf[:, 0] = 3.0   # start token
        prev_surf = start_surf
        hidden = init_hidden

        glass_count = torch.zeros(B, device=device, dtype=torch.long)

        outputs_type = []
        outputs_reg = []
        for t in range(max_len):
            remaining = num_elements - glass_count
            remaining = torch.clamp(remaining, min=0)

            type_logits, reg_pred, hidden = self.forward_step(prev_surf, hidden, cond, remaining)
            outputs_type.append(type_logits.unsqueeze(1))
            outputs_reg.append(reg_pred.unsqueeze(1))

            if targets is not None:
                if t > 0:
                    prev_true_type = targets[:, t-1, 0].long()
                    glass_count += (prev_true_type == 1).long()
                if np.random.random() < teacher_forcing_ratio:
                    prev_surf = targets[:, t, :]
                else:
                    pred_type = type_logits.argmax(dim=-1).float()
                    prev_surf = torch.cat([pred_type.unsqueeze(1), reg_pred], dim=1)
            else:
                pred_type = type_logits.argmax(dim=-1).float()
                glass_count += (pred_type.squeeze() == 1.0).long()
                prev_surf = torch.cat([pred_type.unsqueeze(1), reg_pred], dim=1)

        outputs_type = torch.cat(outputs_type, dim=1)
        outputs_reg = torch.cat(outputs_reg, dim=1)
        return outputs_type, outputs_reg


class LensModel(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=256, cond_dim=128, num_layers=2, max_elements=15, dropout=0.1):
        super().__init__()
        self.encoder = ConditionEncoder(input_dim, hidden_dim, cond_dim, num_layers)
        self.decoder = SurfaceDecoder(cond_dim, hidden_dim, num_layers, dropout=dropout, max_elements=max_elements)

    def forward(self, x, num_elements, targets=None, teacher_forcing_ratio=0.5, max_len=30):
        cond, (h0, c0) = self.encoder(x)
        return self.decoder((h0, c0), cond, targets, teacher_forcing_ratio, max_len, num_elements)


In [6]:
# ================== 3. 损失函数 ==================
def loss_fn_with_physics(type_pred, reg_pred, targets, mask,
                         C_scaler, t_scaler, nd_scaler, vd_scaler,
                         lambda_phys=1.0,
                         t_min=0.1, nd_min=1.0, nd_max=2.0,
                         vd_min=0.0, vd_max=200.0,
                         R_min=0.5, lens_t_min=0.5):
    B, L, _ = targets.shape
    device = type_pred.device

    # 原始任务损失
    type_target = targets[:, :, 0].long()
    ce_loss = F.cross_entropy(type_pred.reshape(-1, 3), type_target.reshape(-1), reduction='none')
    ce_loss = ce_loss.reshape(B, L)
    reg_target = targets[:, :, 1:]
    mse_loss = F.mse_loss(reg_pred, reg_target, reduction='none').mean(dim=-1)

    mask_sum = mask.sum() + 1e-8
    ce_loss = (ce_loss * mask).sum() / mask_sum
    mse_loss = (mse_loss * mask).sum() / mask_sum

    # 反标准化
    C_mean, C_std = C_scaler.mean_[0], C_scaler.scale_[0]
    t_mean, t_std = t_scaler.mean_[0], t_scaler.scale_[0]
    nd_mean, nd_std = nd_scaler.mean_[0], nd_scaler.scale_[0]
    vd_mean, vd_std = vd_scaler.mean_[0], vd_scaler.scale_[0]

    C_pred = reg_pred[..., 0] * C_std + C_mean
    t_pred = reg_pred[..., 1] * t_std + t_mean
    nd_pred = reg_pred[..., 2] * nd_std + nd_mean
    vd_pred = reg_pred[..., 3] * vd_std + vd_mean

    pred_type = type_pred.argmax(dim=-1)

    # 物理惩罚
    is_image = (pred_type == 2)
    is_glass = (pred_type == 1).float()

    t_penalty = torch.relu(t_min - t_pred) ** 2 * (~is_image).float()
    nd_penalty = (torch.relu(nd_min - nd_pred) ** 2 + torch.relu(nd_pred - nd_max) ** 2) * is_glass
    vd_penalty = (torch.relu(vd_min - vd_pred) ** 2 + torch.relu(vd_pred - vd_max) ** 2) * is_glass
    C_abs = torch.abs(C_pred) + 1e-8
    R_pred = 1.0 / C_abs
    R_penalty = torch.relu(R_min - R_pred) ** 2
    glass_t_penalty = torch.relu(lens_t_min - t_pred) ** 2 * is_glass

    phys_penalty = t_penalty + nd_penalty + vd_penalty + R_penalty + glass_t_penalty
    phys_penalty_masked = (phys_penalty * mask.float()).sum() / mask_sum

    total_loss = ce_loss + mse_loss + lambda_phys * phys_penalty_masked
    return total_loss, ce_loss, mse_loss, phys_penalty_masked

In [7]:
# ================== 4. 训练与验证函数（用于Optuna目标） ==================
def train_epoch(model, loader, optimizer, device, dataset, lambda_phys, t_min,
                nd_min, nd_max, vd_min, vd_max, R_min, lens_t_min, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0
    for inp, targets, mask, lengths, num_elements in loader:
        inp = inp.to(device)
        targets = targets.to(device)
        mask = mask.to(device)
        num_elements = num_elements.to(device)

        type_pred, reg_pred = model(inp, num_elements, targets,
                                    teacher_forcing_ratio=teacher_forcing_ratio,
                                    max_len=targets.shape[1])
        loss, _, _, _ = loss_fn_with_physics(type_pred, reg_pred, targets, mask,
                                             dataset.C_scaler, dataset.t_scaler,
                                             dataset.nd_scaler, dataset.vd_scaler,
                                             lambda_phys=lambda_phys,
                                             t_min=t_min, nd_min=nd_min, nd_max=nd_max,
                                             vd_min=vd_min, vd_max=vd_max,
                                             R_min=R_min, lens_t_min=lens_t_min)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * inp.size(0)
    return total_loss / len(loader.dataset)


def validate(model, loader, device, dataset, lambda_phys, t_min,
             nd_min, nd_max, vd_min, vd_max, R_min, lens_t_min):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inp, targets, mask, lengths, num_elements in loader:
            inp = inp.to(device)
            targets = targets.to(device)
            mask = mask.to(device)
            num_elements = num_elements.to(device)

            type_pred, reg_pred = model(inp, num_elements, targets,
                                        teacher_forcing_ratio=0.0,
                                        max_len=targets.shape[1])
            loss, _, _, _ = loss_fn_with_physics(type_pred, reg_pred, targets, mask,
                                                 dataset.C_scaler, dataset.t_scaler,
                                                 dataset.nd_scaler, dataset.vd_scaler,
                                                 lambda_phys=lambda_phys,
                                                 t_min=t_min, nd_min=nd_min, nd_max=nd_max,
                                                 vd_min=vd_min, vd_max=vd_max,
                                                 R_min=R_min, lens_t_min=lens_t_min)
            total_loss += loss.item() * inp.size(0)
    return total_loss / len(loader.dataset)


In [8]:
# ================== 5. Optuna目标函数 ==================
# 全局变量，方便objective访问
global_train_dataset = None
global_val_dataset = None
global_dataset = None
global_device = None
global_max_len = None

def objective(trial):
    # 超参数采样
    num_layers = trial.suggest_int("num_layers", 1, 3)
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    cond_dim = trial.suggest_categorical("cond_dim", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])

    lambda_phys = trial.suggest_float("lambda_phys", 0.1, 2.0)
    t_min = trial.suggest_float("t_min", 0.05, 0.3)

    # 固定物理约束
    nd_min, nd_max = 1.0, 2.0
    vd_min, vd_max = 0.0, 200.0
    R_min = 0.5
    lens_t_min = 0.5

    # 构建数据加载器
    train_loader = DataLoader(global_train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(global_val_dataset, batch_size=batch_size, shuffle=False)

    # 构建模型
    model = LensModel(
        input_dim=7,
        hidden_dim=hidden_dim,
        cond_dim=cond_dim,
        num_layers=num_layers,
        max_elements=15,
        dropout=dropout
    ).to(global_device)

    # 优化器
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "AdamW":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    best_val_loss = float('inf')
    patience = 10
    no_improve = 0
    max_epochs = 100

    for epoch in range(max_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, global_device, global_dataset,
                                 lambda_phys, t_min, nd_min, nd_max, vd_min, vd_max, R_min, lens_t_min,
                                 teacher_forcing_ratio=0.5)
        val_loss = validate(model, val_loader, global_device, global_dataset,
                            lambda_phys, t_min, nd_min, nd_max, vd_min, vd_max, R_min, lens_t_min)

        scheduler.step()

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    return best_val_loss


In [9]:
# ================== 6. 主程序 ==================
# if __name__ == "__main__":
# 加载数据
patent_list = load_data_from_excel('Lensdata.xlsx', 'Surfdata.xlsx')
print(f"成功加载 {len(patent_list)} 个专利设计")

max_surfaces = max(len(patent['surfaces']) for patent in patent_list)
print(f"数据中最大面数：{max_surfaces}")
max_surfaces += 2

dataset = LensDataset(patent_list, max_len=max_surfaces)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 设置全局变量供Optuna使用
global_train_dataset = train_dataset
global_val_dataset = val_dataset
global_dataset = dataset
global_device = device
global_max_len = max_surfaces

# 创建Optuna研究对象
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
print("开始超参数搜索...")
study.optimize(objective, n_trials=25, timeout=3600*6)  # 可根据资源调整

print("\n最佳超参数：")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"最佳验证损失: {study.best_value}")

# 保存最佳超参数
with open("best_hyperparams.json", "w") as f:
    json.dump(study.best_params, f, indent=2)


成功加载 25980 个专利设计
数据中最大面数：69
使用设备: cuda


[I 2026-07-03 16:12:14,693] A new study created in memory with name: no-name-a57642ed-5835-4b44-8145-9ede72bbf430


开始超参数搜索...


[I 2026-07-03 23:49:51,991] Trial 0 finished with value: 1.082585100397135 and parameters: {'num_layers': 3, 'hidden_dim': 256, 'cond_dim': 64, 'dropout': 0.047101740741188725, 'lr': 0.003127808044202295, 'optimizer': 'SGD', 'weight_decay': 7.483223618727563e-06, 'batch_size': 16, 'lambda_phys': 1.3369808202412057, 't_min': 0.24983274799818483}. Best is trial 0 with value: 1.082585100397135.



最佳超参数：
  num_layers: 3
  hidden_dim: 256
  cond_dim: 64
  dropout: 0.047101740741188725
  lr: 0.003127808044202295
  optimizer: SGD
  weight_decay: 7.483223618727563e-06
  batch_size: 16
  lambda_phys: 1.3369808202412057
  t_min: 0.24983274799818483
最佳验证损失: 1.082585100397135


In [10]:
# ---- 使用最佳超参数重新训练最终模型 ----
print("\n使用最佳超参数重新训练最终模型...")
best_params = study.best_params

final_batch_size = best_params["batch_size"]
train_loader = DataLoader(train_dataset, batch_size=final_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=final_batch_size, shuffle=False)

model = LensModel(
    input_dim=7,
    hidden_dim=best_params["hidden_dim"],
    cond_dim=best_params["cond_dim"],
    num_layers=best_params["num_layers"],
    max_elements=15,
    dropout=best_params["dropout"]
).to(device)

optimizer_name = best_params["optimizer"]
if optimizer_name == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
elif optimizer_name == "AdamW":
    optimizer = optim.AdamW(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])
else:
    optimizer = optim.SGD(model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"], momentum=0.9)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500)
best_val_loss = float('inf')
patience = 20
no_improve = 0
epochs = 500
train_losses, val_losses = [], []

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, device, dataset,
                             best_params["lambda_phys"], best_params["t_min"],
                             1.0, 2.0, 0.0, 200.0, 0.5, 0.5,
                             teacher_forcing_ratio=0.5)
    val_loss = validate(model, val_loader, device, dataset,
                        best_params["lambda_phys"], best_params["t_min"],
                        1.0, 2.0, 0.0, 200.0, 0.5, 0.5)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        torch.save(model.state_dict(), 'best_final_model.pth')
    else:
        no_improve += 1

    if epoch % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if no_improve >= patience:
        print(f"早停触发，在 epoch {epoch+1} 停止训练")
        break

# 保存模型和归一化参数
model.load_state_dict(torch.load('best_final_model.pth'))
torch.save(model.state_dict(), 'LSTM_model_tuned.pth')
with open('norm_stats_tuned.pkl', 'wb') as f:
    pickle.dump({
        'input_scaler': dataset.input_scaler,
        'C_scaler': dataset.C_scaler,
        't_scaler': dataset.t_scaler,
        'nd_scaler': dataset.nd_scaler,
        'vd_scaler': dataset.vd_scaler
    }, f)

print("调优完成，最终模型和归一化参数已保存。")


使用最佳超参数重新训练最终模型...
Epoch 1/500 | Train Loss: 3.8069 | Val Loss: 1.6249
Epoch 21/500 | Train Loss: 1.2948 | Val Loss: 1.2458
Epoch 41/500 | Train Loss: 1.1927 | Val Loss: 1.1225
Epoch 61/500 | Train Loss: 1.2597 | Val Loss: 1.0994
Epoch 81/500 | Train Loss: 1.2660 | Val Loss: 0.9784
Epoch 101/500 | Train Loss: 1.4758 | Val Loss: 0.9461
Epoch 121/500 | Train Loss: 1.5146 | Val Loss: 0.9355
Epoch 141/500 | Train Loss: 1.0127 | Val Loss: 0.9058
Epoch 161/500 | Train Loss: 1.0409 | Val Loss: 0.8742
Epoch 181/500 | Train Loss: 1.9283 | Val Loss: 0.8660
Epoch 201/500 | Train Loss: 0.9500 | Val Loss: 0.8404
Epoch 221/500 | Train Loss: 0.9371 | Val Loss: 0.8264
Epoch 241/500 | Train Loss: 0.9312 | Val Loss: 0.8163
Epoch 261/500 | Train Loss: 0.8632 | Val Loss: 0.8097
Epoch 281/500 | Train Loss: 0.8992 | Val Loss: 0.8035
Epoch 301/500 | Train Loss: 0.8733 | Val Loss: 0.7886
Epoch 321/500 | Train Loss: 0.8982 | Val Loss: 0.7788
Epoch 341/500 | Train Loss: 0.8455 | Val Loss: 0.7731
Epoch 361/500 

In [11]:
# ================== 7. 推理（生成表面序列，修正版） ==================
def predict(model, input_features, dataset, num_elements, max_len=30, device='cpu'):
    """
    input_features: numpy array or list of shape (7,)
    dataset: LensDataset 实例（用于反标准化）
    num_elements: int，期望的元件数（玻璃数）
    返回: list of dict 表面序列
    """
    model.eval()
    # 标准化输入
    inp = torch.FloatTensor(dataset.input_scaler.transform([input_features])).to(device)
    cond, (h0, c0) = model.encoder(inp)          # cond: (1, cond_dim), h0,c0: (1,1,hidden_dim)
    cond = cond.unsqueeze(1)                     # (1, 1, cond_dim)

    B = 1
    start_surf = torch.zeros(B, 5, device=device)
    start_surf[:, 0] = 3.0                       # start token
    prev_surf = start_surf
    hidden = (h0, c0)

    glass_count = 0
    surfaces = []
    for t in range(max_len):
        remaining = num_elements - glass_count
        remaining = max(remaining, 0)
        rem_tensor = torch.tensor([remaining], dtype=torch.float, device=device)

        type_logits, reg_pred, hidden = model.decoder.forward_step(
            prev_surf, hidden, cond, rem_tensor
        )
        type_idx = type_logits.argmax(dim=-1).item()   # 0:air, 1:glass, 2:image
        type_name = idx_to_type[type_idx]

        # 反标准化
        C = dataset.C_scaler.inverse_transform([[reg_pred[0, 0].item()]])[0, 0]
        t_val = dataset.t_scaler.inverse_transform([[reg_pred[0, 1].item()]])[0, 0]
        nd = dataset.nd_scaler.inverse_transform([[reg_pred[0, 2].item()]])[0, 0]
        vd = dataset.vd_scaler.inverse_transform([[reg_pred[0, 3].item()]])[0, 0]

        # 根据类型强制修正（物理合理性）
        if type_name in ['air', 'image']:
            nd, vd = 0.0, 0.0
        if type_name == 'image':
            C, t_val = 0.0, 0.0

        surfaces.append({
            'type': type_name,
            'C': C,
            't': t_val,
            'nd': nd,
            'vd': vd
        })

        # 更新已生成的玻璃数
        if type_name == 'glass':
            glass_count += 1

        # 准备下一步输入
        next_surf = torch.zeros(1, 5, device=device)
        next_surf[0, 0] = float(type_idx)
        next_surf[0, 1] = dataset.C_scaler.transform([[C]])[0, 0]
        next_surf[0, 2] = dataset.t_scaler.transform([[t_val]])[0, 0]
        next_surf[0, 3] = dataset.nd_scaler.transform([[nd]])[0, 0] if nd != 0 else 0.0
        next_surf[0, 4] = dataset.vd_scaler.transform([[vd]])[0, 0] if vd != 0 else 0.0
        prev_surf = next_surf

        if type_name == 'image':
            break

    return surfaces

In [17]:
# ================== 8. 预计新设计 ==================
my_spec = [
    100.0,      # EFL
    2.0,        # F/#
    20.0,       # Max Field Angle
    20.0,       # Image Height
    50.0,       # BFL
    7,          # Number of Elements
    0.5876      # Primary Wavelength
]
num_elems = int(my_spec[5])   # 5个元件
surfaces = predict(model, my_spec, dataset, num_elems, max_len=30, device=device)

print("\n生成的光学设计表面序列：")
for i, s in enumerate(surfaces):
    print(f"面{i:2d}: type={s['type']:5s} C={s['C']:8.4f} t={s['t']:8.4f} nd={s['nd']:.4f} vd={s['vd']:.2f}")


生成的光学设计表面序列：
面 0: type=air   C=  0.0510 t=476.4941 nd=0.0000 vd=0.00
面 1: type=glass C=  0.4552 t=134.1254 nd=1.0998 vd=37.70
面 2: type=glass C=  0.1677 t= 96.3791 nd=1.2038 vd=33.60
面 3: type=glass C=  0.3104 t= 73.1329 nd=1.2435 vd=34.68
面 4: type=air   C=  0.5519 t= 62.4345 nd=0.0000 vd=0.00
面 5: type=glass C=  0.6155 t= 63.2239 nd=1.2202 vd=34.14
面 6: type=air   C=  0.6409 t= 68.6279 nd=0.0000 vd=0.00
面 7: type=glass C=  0.4859 t= 65.2173 nd=1.1366 vd=32.12
面 8: type=air   C=  0.4518 t= 95.1233 nd=0.0000 vd=0.00
面 9: type=glass C=  0.0367 t=100.3172 nd=0.9200 vd=27.82
面10: type=glass C= -0.0321 t=114.3419 nd=0.8354 vd=29.84
面11: type=image C=  0.0000 t=  0.0000 nd=0.0000 vd=0.00
